IMPORTS

In [1]:
import json
from typing import List

# -------------------------
# UNSTRUCTURED IMPORTS
# -------------------------
# For PDF partitioning
from unstructured.partition.pdf import partition_pdf
# For hierarchical chunking
from unstructured.chunking.title import chunk_by_title

# -------------------------
# LANGCHAIN IMPORTS
# -------------------------
from langchain_core.documents import Document

# Ollama LLM + Embeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings

# Vector DB (Chroma)
from langchain_chroma import Chroma

# Messages for LLM calls
from langchain_core.messages import HumanMessage

# Environment variables
from dotenv import load_dotenv


c:\Users\LOQ\Downloads\MULTIMODEL_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PARTITION OF PDF INTO ATOMIC ELEMENTS

In [2]:
# ==========================================================
# 1. PARTITION PDF
# ==========================================================
def partition_document(file_path: str):
    print(f"\n📄 Partitioning: {file_path}\n")
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True,
    )
    print(f"✅ Extracted {len(elements)} elements")
    return elements

In [14]:
elements

CHUNKING BY TITLE

In [3]:
# ==========================================================
# 2. CHUNK BY TITLE
# ==========================================================
def create_chunks(elements):
    print("\n🔹 Creating hierarchical title-aware chunks...\n")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

In [12]:
chunks

In [13]:
chunks[2].to_dict()

{'type': 'CompositeElement',
 'element_id': 'b51b9df1-7258-4dd3-aec2-40eb4ab248ab',
 'text': '1. Introduction\n\nIn recent years, advances in digital dentistry have led to the increasing use of intraoral scanners in clinical practice. Compared to conventional impression techniques, these scanners enable faster, more comfortable, and highly accurate data acquisition. They also facilitate orthodontic diagnosis and treatment planning by allowing for the rapid and\n\nAppl. Sci. 2025, 15, 7786\n\nhttps://doi.org/10.3390/app15147786\n\nAppl. Sci. 2025, 15, 7786\n\nseamless transfer of digital data, instant access to patient records, and a reduced need for physical storage [1,2].\n\nAlongside these developments, the integration of digital technologies into orthodontics has enabled more sophisticated planning tools. Digital software platforms now allow for the simulation and visualization of treatment outcomes in three dimensions, significantly enhancing the precision of diagnosis and treatmen

USE LLM SUMMARIZE AND CHNGE TO LANCHAIN DOCS

In [4]:
# ==========================================================
# 3. SEPARATE TEXT / TABLES / IMAGES
# ==========================================================
def separate_content_types(chunk):
    content_data = {
        "text": chunk.text,
        "tables": [],
        "images": [],
        "types": ["text"]
    }

    if hasattr(chunk.metadata, "orig_elements"):
        for element in chunk.metadata.orig_elements:
            block_type = getattr(element, "category", None)

            # ----- TABLE DETECTION -----
            if block_type == "Table":
                content_data["types"].append("table")
                table_html = getattr(element.metadata, "text_as_html", element.text)
                content_data["tables"].append(table_html)

            # ----- IMAGE DETECTION -----
            if block_type == "Image":
                img = getattr(element.metadata, "image_base64", None)
                if img:
                    content_data["types"].append("image")
                    content_data["images"].append(img)

    content_data["types"] = list(set(content_data["types"]))
    return content_data

In [10]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]):
    try:
        llm = ChatOllama(model="llama3.2:3b")

        # Build prompt as plain text
        prompt_text = "You are creating a searchable description for semantic retrieval.\n\n"
        prompt_text += "TEXT CONTENT:\n"
        prompt_text += text + "\n\n"

        if tables:
            prompt_text += "TABLE CONTENT:\n"
            for i, table in enumerate(tables):
                prompt_text += f"[Table {i+1}]\n{table}\n\n"

        if images:
            prompt_text += f"[{len(images)} images included but not displayed]\n\n"

        prompt_text += """
TASK:
Generate a comprehensive, detailed, searchable description including:

- Key facts & numbers
- Main topics and findings
- Relationships between concepts
- Important conclusions
- Anything useful for retrieval

Return ONLY the enhanced description.
"""

        # Send plain string to the model — required for Ollama
        response = llm.invoke(prompt_text)

        return response.content

    except Exception as e:
        print(f"❌ Error creating AI summary: {e}")
        return text


In [6]:
def summarise_chunks(chunks):
    langchain_docs = []

    for chunk in chunks:
        content_data = separate_content_types(chunk)

        # AI-enhanced summary if media exists
        if content_data["tables"] or content_data["images"]:
            enhanced = create_ai_enhanced_summary(
                content_data["text"],
                content_data["tables"],
                content_data["images"]
            )
        else:
            enhanced = content_data["text"]

        # Flatten metadata
        doc = Document(
            page_content=enhanced,
            metadata={
                "raw_text": content_data["text"],
                "tables_html": json.dumps(content_data["tables"]),
                "images_base64": json.dumps(content_data["images"]),
            }
        )

        langchain_docs.append(doc)

    print(f"\n📦 Processed {len(langchain_docs)} chunks\n")
    return langchain_docs


In [15]:
processed_chunks

[Document(metadata={'raw_text': 'it applied sciences\n\n(mbPr\n\nArticle\n\nDiagnostic Accuracy and Agreement Between AI and Clinicians in Orthodontic 3D Model Analysis\n\nSabahattin Bor , Fırat O˘guz * and Ayla Khanmohammadi\n\nDepartment of Orthodontics, Faculty of Dentistry, ˙Inönü University, Malatya 44280, Türkiye; venaroshan@gmail.com (S.B.); ayla_dr@yahoo.com (A.K.)\n\n* Correspondence: firat.oguz.3408@gmail.com\n\nAbstract\n\ncheck for updates\n\nAcademic Editor: Iole Vozza\n\nReceived: 6 June 2025 Revised: 8 July 2025 Accepted: 9 July 2025 Published: 11 July 2025\n\nBackground: Artificial intelligence (AI) is increasingly integrated into orthodontic work- flows, including digital model analysis modules embedded in orthodontic software. While these systems offer efficiency and automation, the accuracy and clinical reliability of AI- generated measurements and diagnostic assessments remain unclear. Therefore, to use AI systems safely and effectively in clinical orthodontics, it 

Put into Vector DB

In [7]:
# ==========================================================
# 6. CREATE VECTOR STORE
# ==========================================================
def create_vector_store(documents, persist_directory="db/chroma_db"):
    print("🔹 Creating embedding model (mxbai-embed-large)...")
    embedding_model = OllamaEmbeddings(model="mxbai-embed-large")

    print("🔹 Creating vector store...")
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_name="medical_rag"
    )

    print(f"✅ Vector store created at {persist_directory}")
    return vector_store


In [8]:
# ==========================================================
# 7. EXPORT RESULTS
# ==========================================================
def export_chunks_to_json(chunks, filename="rag_results.json"):
    data = [
        {"id": i, "content": c.page_content, "metadata": c.metadata}
        for i, c in enumerate(chunks)
    ]

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"✅ Saved RAG results to {filename}")

In [ ]:
print(db._collection.count())

In [11]:
# ==========================================================
# EXECUTION PIPELINE — CORRECT ORDER
# ==========================================================

file_path = "docs/Diagnostic Accuracy and Agreement Between AI and Clinicians in Orthodontic 3D Model Analysis.pdf"

elements = partition_document(file_path)
chunks = create_chunks(elements)
processed_chunks = summarise_chunks(chunks)

db = create_vector_store(processed_chunks)

query = "What are the main findings of the orthodontic AI diagnostic accuracy study?"

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})
results = retriever.invoke(query)

export_chunks_to_json(results, "rag_results.json")


📄 Partitioning: docs/Diagnostic Accuracy and Agreement Between AI and Clinicians in Orthodontic 3D Model Analysis.pdf

✅ Extracted 280 elements

🔹 Creating hierarchical title-aware chunks...

✅ Created 25 chunks

📦 Processed 25 chunks

🔹 Creating embedding model (mxbai-embed-large)...
🔹 Creating vector store...
✅ Vector store created at db/chroma_db
✅ Saved RAG results to rag_results.json
